In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_parquet("train.parquet")

In [3]:
data.shape

(1639424, 7)

In [4]:
data.describe()

,Date,X1,X2,X3,X4,X5
count,1639424,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06
mean,2022-12-03 07:23:43.817145600,1.139258e+00,5.488189e+00,4.110388e+32,2.706323e+29,1.187219e+00
min,2020-12-16 00:00:00,1.000000e+00,5.412539e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,2021-12-10 00:00:00,1.049171e+00,5.480597e+00,1.000000e+00,1.000000e+00,0.000000e+00
50%,2022-11-30 00:00:00,1.105171e+00,5.488979e+00,1.000000e+00,1.000000e+00,6.931472e-01
75%,2023-11-23 00:00:00,1.214096e+00,5.496717e+00,1.000000e+00,2.718282e+00,2.890372e+00
max,2024-12-11 00:00:00,4.014850e+00,5.541852e+00,1.651636e+38,5.540622e+34,3.465736e+00
std,NaN,1.391992e-01,1.342811e-02,2.346156e+35,5.812988e+31,1.304814e+00


In [5]:
data['X3'].describe()

count    1.639424e+06
mean     4.110388e+32
std      2.346156e+35
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.651636e+38
Name: X3, dtype: float64

In [6]:
print(data['X3'].mean() + 3*data['X3'].std())
print(data['X3'].mean() - 3*data['X3'].std())


7.042577611608274e+35
-7.034356834631367e+35


In [7]:
df_test = pd.read_parquet("test.parquet")


In [8]:
df_test.shape

(409856, 7)

In [9]:
data.iloc[4995:5000]


,Date,X1,X2,X3,X4,X5,target
4995,2020-12-21,1.226298,5.496471,1.0,1.0,2.944439,0
4996,2020-12-21,1.218962,5.491084,1.0,1.0,2.944439,0
4997,2020-12-21,1.218962,5.491538,1.0,1.0,2.890372,0
4998,2020-12-21,1.096365,5.492691,1.0,1.0,2.944439,0
4999,2020-12-21,1.000000,5.493720,1.0,1.0,2.944439,0


In [10]:
(data["X3"] <= 1).sum() #count a=how many value are less than 1 in x3


np.int64(1504150)

In [11]:
(data["X4"] <= 1).sum()


np.int64(1149739)

In [12]:
data["Date"].nunique()


1432

In [13]:
data.isna().sum()

Date      0
X1        0
X2        0
X3        0
X4        0
X5        0
target    0
dtype: int64

In [14]:
def preprocess(df):
    df = df.copy()
    
    # Date parsing
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Basic time features
    df["year"] = df["Date"].dt.year
    df["month"] = df["Date"].dt.month
    df["day"] = df["Date"].dt.day
    df["dayofweek"] = df["Date"].dt.dayofweek
    
    # Extra strong date features (0.83+ booster)
    df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = df["Date"].dt.is_month_start.astype(int)
    df["is_month_end"] = df["Date"].dt.is_month_end.astype(int)

    # Log transforms (simple + safe)
    df["X3"] = np.log1p(np.log1p(np.log1p(df["X3"])))
    df["X4"] = np.log1p(np.log1p(np.log1p(df["X4"])))
    
    # Drop Date
    df.drop(["Date"], axis=1, inplace=True)
    
    return df

In [15]:
train_fe = preprocess(data)
test_fe  = preprocess(df_test)

print(train_fe.head())
print(test_fe.head())


         X1        X2        X3        X4        X5 target  year  month  day  \
0  1.518921  5.463154  0.423036  0.609036  2.890372      0  2020     12   16   
1  1.546509  5.458010  0.423036  0.609036  2.833213      1  2020     12   16   
2  1.645427  5.456560  0.423036  0.760830  2.890372      1  2020     12   16   
3  1.652022  5.458479  0.423036  0.609036  2.890372      1  2020     12   16   
4  1.695538  5.466709  0.423036  0.609036  2.890372      0  2020     12   16   

   dayofweek  weekofyear  is_weekend  is_month_start  is_month_end  
0          2          51           0               0             0  
1          2          51           0               0             0  
2          2          51           0               0             0  
3          2          51           0               0             0  
4          2          51           0               0             0  
   ID        X1        X2        X3        X4        X5  year  month  day  \
0   0  1.685395  5.463917  0

In [16]:
train_fe.describe()


,X1,X2,X3,X4,X5,year,month,day,dayofweek,weekofyear,is_weekend,is_month_start,is_month_end
count,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06,1.639424e+06
mean,1.139258e+00,5.488189e+00,4.677789e-01,5.184130e-01,1.187219e+00,2.022425e+03,6.495986e+00,1.566405e+01,2.997283e+00,2.646790e+01,2.869563e-01,3.344955e-02,3.249251e-02
std,1.391992e-01,1.342811e-02,1.925555e-01,2.028125e-01,1.304814e+00,1.124472e+00,3.446579e+00,8.836483e+00,2.005976e+00,1.505039e+01,4.523411e-01,1.798074e-01,1.773042e-01
min,1.000000e+00,5.412539e+00,4.230359e-01,4.230359e-01,0.000000e+00,2.020000e+03,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.049171e+00,5.480597e+00,4.230359e-01,4.230359e-01,0.000000e+00,2.021000e+03,4.000000e+00,8.000000e+00,1.000000e+00,1.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.105171e+00,5.488979e+00,4.230359e-01,4.230359e-01,6.931472e-01,2.022000e+03,7.000000e+00,1.600000e+01,3.000000e+00,2.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.214096e+00,5.496717e+00,4.230359e-01,6.090362e-01,2.890372e+00,2.023000e+03,1.000000e+01,2.300000e+01,5.000000e+00,4.000000e+01,1.000000e+00,0.000000e+00,0.000000e+00
max,4.014850e+00,5.541852e+00,1.702680e+00,1.685370e+00,3.465736e+00,2.024000e+03,1.200000e+01,3.100000e+01,6.000000e+00,5.300000e+01,1.000000e+00,1.000000e+00,1.000000e+00


Further Code cells deal with the Scaling and modelling of the data.¶


In [17]:
!pip3  install xgboost

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [19]:
data["target"] = data["target"].astype(int)


In [20]:
from sklearn.preprocessing import RobustScaler

def Scaler(df,X_column_list):
    X = df[X_column_list]

    scaler = RobustScaler()

    X_scaled = scaler.fit_transform(X)

    X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

    return scaler, X_scaled_df

In [21]:
train_fe.head()

,X1,X2,X3,X4,X5,target,year,month,day,dayofweek,weekofyear,is_weekend,is_month_start,is_month_end
0,1.518921,5.463154,0.423036,0.609036,2.890372,0,2020,12,16,2,51,0,0,0
1,1.546509,5.458010,0.423036,0.609036,2.833213,1,2020,12,16,2,51,0,0,0
2,1.645427,5.456560,0.423036,0.760830,2.890372,1,2020,12,16,2,51,0,0,0
3,1.652022,5.458479,0.423036,0.609036,2.890372,1,2020,12,16,2,51,0,0,0
4,1.695538,5.466709,0.423036,0.609036,2.890372,0,2020,12,16,2,51,0,0,0


In [22]:
X_column_list = ["X1", "X2", "X3", "X4", "X5","year", "month","day", "dayofweek","weekofyear","is_weekend","is_month_start","is_month_end"]
scaler, df_train_scaled = Scaler(train_fe,X_column_list)

In [23]:
train_fe = pd.concat([df_train_scaled.reset_index(drop=True), data["target"].reset_index(drop=True)], axis=1)

In [24]:

X = train_fe.iloc[:,:-1]  # your features
y = train_fe["target"] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [25]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[324902    175]
 [   791   2017]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    325077
           1       0.92      0.72      0.81      2808

    accuracy                           1.00    327885
   macro avg       0.96      0.86      0.90    327885
weighted avg       1.00      1.00      1.00    327885



In [26]:
!python.exe -m pip install --upgrade pip
!pip install lightgbm

In [27]:
X_test_final = test_fe[["X1", "X2", "X3", "X4", "X5","year", "month","day", "dayofweek","weekofyear","is_weekend","is_month_start","is_month_end"]]

In [ ]:
X_test_final_scaled = scaler.transform(X_test_final)
X_test_final = pd.DataFrame(X_test_final_scaled, columns=X_test_final.columns)
X_test = X_test[X_train.columns]
test_preds = model.predict(X_test)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probs = np.zeros(len(X))
test_probs_avg = np.zeros(len(X_test_final_scaled))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    neg = (y_tr==0).sum()
    pos = (y_tr==1).sum()
    spw = neg/pos

    lgb = LGBMClassifier(
        n_estimators=7000,
        learning_rate=0.02,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=60,
        reg_lambda=2.0,
        scale_pos_weight=spw,
        random_state=42
    )

    lgb.fit(X_tr, y_tr)

    # OOF probs
    oof_probs[va_idx] = lgb.predict_proba(X_va)[:,1]

    # Test probs avg
    test_probs_avg += lgb.predict_proba(X_test_final_scaled)[:,1] / 5

    print(f"Fold {fold} done")


In [ ]:
best_f1, best_t = 0, 0.5
for t in np.arange(0.90, 0.95, 0.002):
    oof_pred = (oof_probs > t).astype(int)
    score = f1_score(y, oof_pred, average="binary")
    if score > best_f1:
        best_f1, best_t = score, t

print(" OOF Best F1:", best_f1)
print(" Best Threshold:", best_t)


In [ ]:
final_preds = (test_probs_avg > best_t).astype(int)
8173
submission = pd.DataFrame({
    "ID": test_fe["ID"],
    "target": final_preds.astype(int)
})

submission.to_parquet("Submission_CV_LGB.parquet", index=False)

print(" Saved Submission_CV_LGB.parquet")
print("Predicted 1s:", submission["target"].sum())

In [ ]:
oof1, test1 = train_lgb_cv(X, y, X_test, seed=42)
oof2, test2 = train_lgb_cv(X, y, X_test, seed=99)

# Blend LGBM seeds
oof_lgb = 0.5*oof1 + 0.5*oof2
test_lgb = 0.5*test1 + 0.5*test2


In [ ]:
test_fe.head()

In [ ]:
X_test = test_fe.drop("ID", axis=1)  # keep ID for submission
test_ids = test_fe["ID"].values
print(X_test.shape)
print(test_ids)

print(X_test_final_scaled.shape)

In [ ]:
print("len(test_ids):", len(test_ids))
print("len(test_lgb):", len(test_lgb))
print("X_test shape:", X_test.shape)


In [ ]:
final_preds = (test_lgb > best_t).astype(int)

submission = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds.astype(int)
})

submission.to_parquet("Submission_FINAL_LGB.parquet", index=False)
print("Saved: Submission_FINAL_LGB.parquet")
print("Predicted 1s:", submission["target"].sum())


In [ ]:
def cv_probs(seed):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    testp = np.zeros(len(X_test_final_scaled))

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        neg = (y_tr==0).sum()
        pos = (y_tr==1).sum()
        spw = neg/pos

        lgb = LGBMClassifier(
            n_estimators=8000,
            learning_rate=0.02,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=60,
            reg_lambda=2.0,
            scale_pos_weight=spw,
            random_state=seed
        )

        lgb.fit(X_tr, y_tr)
        oof[va_idx] = lgb.predict_proba(X_va)[:,1]
        testp += lgb.predict_proba(X_test_final_scaled)[:,1] / 5
        print(f" Seed {seed} Fold {fold} done")

    return oof, testp

In [ ]:

oof42, test42 = cv_probs(42)
oof99, test99 = cv_probs(99)
oof2024, test2024 = cv_probs(2024)

oof_final = (oof42 + oof99 + oof2024) / 3
test_final = (test42 + test99 + test2024) / 3

In [ ]:
data.to_csv("anaverse_output.csv", index=False)
